Create the required python environment using the requirements.txt

In [ ]:
import numpy as np
from bioio_ome_zarr.writers import OMEZarrWriter
import dask.array as da
from typing import List, Optional
from bioio import BioImage
import napari

This script reads in a pyramidal CZI file at full resolution, calculates the downsampling required for creating a pyramidal Zarr array, and then writes it out. 

Does not add in or preserve metadata in current state. Metadata can be added in if needed.

In [4]:
def calculate_pyramid_shapes(
    base_shape: tuple,
    num_channels: int,
    num_levels: int = 5,
    downsample_factors: Optional[List[int]] = None
) -> List[tuple]:
    """
    Calculate shapes for each pyramid level.

    Parameters
    ----------
    base_shape : tuple
        Shape of the full resolution image (Y, X)
    num_channels : int
        Number of channels
    num_levels : int, optional
        Number of pyramid levels (default: 5)
    downsample_factors : list of int, optional
        Custom downsample factors (default: [1, 2, 4, 8, 12])

    Returns
    -------
    list of tuple
        List of shapes (C, Y, X) for each pyramid level
    """
    if downsample_factors is None:
        downsample_factors = [1, 2, 4, 8, 12]

    # Ensure we have the right number of factors
    if len(downsample_factors) != num_levels:
        # Generate factors
        downsample_factors = [2**i if i < 4 else 12 for i in range(num_levels)]

    level_shapes = []
    for factor in downsample_factors:
        y_size = int(base_shape[0] / factor)
        x_size = int(base_shape[1] / factor)
        level_shapes.append((num_channels, y_size, x_size))

    return level_shapes

In [ ]:
#read in the image data
tma = BioImage(r'Path\to\CZI_Image.czi')
tma_dask = tma.get_dask_stack()
tma_dask = da.squeeze(tma_dask) #remove empty dims common in czi files
h_flip = np.flip(tma_dask,axis=2) #apply horizontal flip

In [ ]:
#get pixel size for use later
pixel_sizes = tma.physical_pixel_sizes
physical_pixel_size = [
    0,  # C dimension has no physical size
    pixel_sizes.Y if pixel_sizes.Y else 1.0,
    pixel_sizes.X if pixel_sizes.X else 1.0
]

print(f"  Physical pixel size: Y={physical_pixel_size[1]}, X={physical_pixel_size[2]} µm")

In [ ]:
#Can check that horizontal flip is correctly applied:
viewer = napari.Viewer()
viewer.add_image(tma_dask[0,:,:], name='original') #just using the first channel as reference (DAPI)
viewer.add_image(h_flip[0,:,:],name="flipped")

In [ ]:
# Calculate pyramid level shapes
base_shape = h_flip.shape[1:]  # (Y, X)
level_shapes = calculate_pyramid_shapes(
    base_shape,
    h_flip.shape[0]
)

Note that QuPath v6.0 requires a separate OME file contained within the zarr array to read in metadata. 

This script does not include creating this extra metadata file as the relevant metadata (e.g., pixel size, magnification, and channel names) can be added through QuPath.

In [ ]:
output_path = r"Path\to\save\name.zarr"
writer = OMEZarrWriter(
        store=str(output_path),
        level_shapes=level_shapes,
        dtype=h_flip.dtype,
        zarr_format=2,
        chunk_shape=[4,512,512],
        axes_names=["c", "y", "x"],
        axes_types=["channel", "space", "space"],
        axes_units=[None, "micrometer", "micrometer"],
        physical_pixel_size=physical_pixel_size
    )

# Write the data
print(f"  Writing data to Zarr (this may take a while)...")
writer.write_full_volume(h_flip)